# NVIDIA Nemotron Reasoning Challenge — Submission

This notebook runs inference on the test set using a fine-tuned Nemotron-3-Nano-30B-A3B model.

**GPU**: G4 with 96GB VRAM — loads full BF16 model via vLLM.

**Strategy**: maj@64 majority voting across 64 generations per prompt.

In [ ]:
# Install dependencies
!pip install -q vllm>=0.7.0 transformers pandas

In [ ]:
import os
import re
import math
import pandas as pd
from collections import Counter
from vllm import LLM, SamplingParams

# Configuration
MODEL_PATH = "/kaggle/input/your-model"  # TODO: Update with your model path
TEST_CSV = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/test.csv"
NUM_GENERATIONS = 64
TEMPERATURE = 1.0
TOP_P = 1.0
MAX_NEW_TOKENS = 2048

In [ ]:
# Load model with vLLM
llm = LLM(
    model=MODEL_PATH,
    trust_remote_code=True,
    dtype="auto",
    max_model_len=8192,
    gpu_memory_utilization=0.92,
)
tokenizer = llm.get_tokenizer()
print(f"Model loaded: {MODEL_PATH}")

In [ ]:
# Load test data
test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")
print(test_df.head())

In [ ]:
# Category detection
def detect_category(prompt):
    prompt_lower = prompt.lower()
    if "bit manipulation" in prompt_lower:
        return "bit_manipulation"
    elif "gravitational" in prompt_lower:
        return "gravitational_constant"
    elif "unit conversion" in prompt_lower:
        return "unit_conversion"
    elif "encryption" in prompt_lower or "decrypt" in prompt_lower:
        return "text_encryption"
    elif "numeral system" in prompt_lower:
        return "numeral_system"
    elif "transformation rules" in prompt_lower:
        return "equation_transformation"
    return "unknown"

test_df["category"] = test_df["prompt"].apply(detect_category)
print(test_df["category"].value_counts())

In [ ]:
# Answer extraction utilities
def extract_answer(response, category):
    """Extract and normalize answer from model response."""
    # Remove <think>...</think> block
    cleaned = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    if not cleaned:
        cleaned = response.strip()
    
    # Category-specific normalization
    if category == "bit_manipulation":
        match = re.search(r"[01]{8}", cleaned)
        return match.group(0) if match else cleaned.strip()
    elif category in ("gravitational_constant", "unit_conversion"):
        match = re.search(r"-?\d+\.?\d*", cleaned)
        return match.group(0) if match else cleaned.strip()
    elif category == "text_encryption":
        return cleaned.strip().lower()
    elif category == "numeral_system":
        return cleaned.strip().upper()
    else:
        return cleaned.strip()


def match_answers(a, b, category):
    """Category-aware answer matching for majority vote clustering."""
    if category in ("gravitational_constant", "unit_conversion"):
        try:
            va, vb = float(a), float(b)
            if vb == 0:
                return abs(va) < 1e-6
            return abs(va - vb) / abs(vb) <= 0.01
        except ValueError:
            return a == b
    elif category == "text_encryption":
        return a.lower() == b.lower()
    else:
        return a == b


def majority_vote(responses, category):
    """Find majority answer with category-aware clustering."""
    answers = [extract_answer(r, category) for r in responses]
    answers = [a for a in answers if a]  # filter empty
    if not answers:
        return ""
    
    if category in ("gravitational_constant", "unit_conversion"):
        # Cluster by approximate equality
        clusters = []
        for ans in answers:
            placed = False
            for cluster in clusters:
                if match_answers(ans, cluster[0], category):
                    cluster.append(ans)
                    placed = True
                    break
            if not placed:
                clusters.append([ans])
        best = max(clusters, key=len)
        return best[0]
    else:
        counter = Counter(answers)
        return counter.most_common(1)[0][0]

In [ ]:
# Build prompts with chat template
sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
)

# Format prompts
formatted_prompts = []
for _, row in test_df.iterrows():
    messages = [
        {"role": "system", "content": "detailed thinking on"},
        {"role": "user", "content": row["prompt"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    formatted_prompts.append(text)

print(f"Formatted {len(formatted_prompts)} prompts")
print(f"Sample prompt length: {len(formatted_prompts[0])} chars")

In [ ]:
# Run inference with maj@64
results = []

for idx, (_, row) in enumerate(test_df.iterrows()):
    prompt = formatted_prompts[idx]
    category = row["category"]
    
    # Generate N responses
    expanded = [prompt] * NUM_GENERATIONS
    outputs = llm.generate(expanded, sampling_params)
    responses = [o.outputs[0].text for o in outputs]
    
    # Majority vote
    answer = majority_vote(responses, category)
    results.append({"id": row["id"], "answer": answer})
    
    print(f"[{idx+1}/{len(test_df)}] {category}: {answer[:50]}")

print(f"\nCompleted inference for {len(results)} samples")

In [ ]:
# Write submission
submission = pd.DataFrame(results)
submission.to_csv("submission.csv", index=False)
print(f"Submission saved: {len(submission)} rows")
print(submission.head(10))